# Preprocessing Pipeline for YOLO Training

## Preprocessing Steps Explained

- **Letterbox Resizing**  
  Resize each image to a fixed square size (e.g. 640×640) without distortion by scaling to fit and padding the shorter edges. Ensures all inputs share the same dimensions while preserving object shapes and centering content.

- **CLAHE (Contrast-Limited Adaptive Histogram Equalization)**  
  Enhances local contrast by applying histogram equalization on small image tiles, then clipping extreme amplification to avoid noise blow-up. Makes faint features (like flagellar motors) more visible in low-contrast tomogram slices.

- **NLMeans Denoising**  
  Reduces speckle and random noise by averaging each patch with similar patches found across the image, preserving textures and edges. Particularly effective for the grainy appearance of cryo-ET data.

- **Grayscale Normalization**  
  Stretches pixel intensities to the full 0–255 range, standardizing brightness and contrast across slices. A step before further enhancement and denoising.

Each of these operations boosts the signal-to-noise and ensures uniform, centered inputs for a YOLO detector, improving both training stability and detection accuracy.```


## References

[Data Preprocessing](https://docs.ultralytics.com/guides/preprocessing_annotated_data/)

[Data Augmentation](https://www.ultralytics.com/glossary/data-augmentation)



In [ ]:
import os
import sys
from tqdm import tqdm
import random
import shutil
import pandas as pd
import albumentations as A
import cv2

# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config


In [24]:
#  Imports & Configuration

# User parameters

if config.USE_SAMPLED_TRAIN_DATASET:
    INPUT_DIR    = config.SAMPLED_TRAIN_DATASET_DIR         # path of sampled training dataset
else:
    INPUT_DIR    = config.TRAIN_DATASET_DIR                  # path of training dataset

PREPROCESSED_OUTPUT_DIR   = config.PREPROCESSED_DATASET_DIR
TARGET_SIZE  = (640, 640)                          # (height, width)
EXTS         = [".jpg", ".png", ".tif", ".tiff"]   # supported extensions

# CLAHE & denoise settings
CLAHE_CFG    = {"clip_limit": 2.0, "grid_size": (8, 8)}
DENOISE_CFG  = {"h": 10, "template_size": 7, "search_size": 21}

# Ensure output directory exists
os.makedirs(PREPROCESSED_OUTPUT_DIR, exist_ok=True)


In [25]:
# Helper Functions


def letterbox_resize(img, new_shape=TARGET_SIZE, color=(114,114,114)):
    """
    Resize+pad to new_shape, keeping aspect ratio (letterbox).
    """

    h0, w0 = img.shape[:2]
    r = min(new_shape[0]/h0, new_shape[1]/w0)
    new_unpad = (int(w0*r), int(h0*r))
    dh, dw = new_shape[0] - new_unpad[1], new_shape[1] - new_unpad[0]
    dh, dw = dh/2, dw/2

    img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh-0.1)), int(round(dh+0.1))
    left, right = int(round(dw-0.1)), int(round(dw+0.1))

    if len(img.shape)==2:
        return cv2.copyMakeBorder(img, top, bottom, left, right,
                                  cv2.BORDER_CONSTANT, value=color[0])
    else:
        return cv2.copyMakeBorder(img, top, bottom, left, right,
                                  cv2.BORDER_CONSTANT, value=color)

def apply_clahe(gray, clip_limit, grid_size):
    """
    Apply CLAHE on a grayscale image.
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    return clahe.apply(gray)

def denoise_image(gray, h, template_size, search_size):
    """
    Denoise grayscale with NLMeans.
    """
    return cv2.fastNlMeansDenoising(gray, None, h, template_size, search_size)

def preprocess_image(img,
                     clahe_cfg=CLAHE_CFG,
                     denoise_cfg=DENOISE_CFG,
                     target_size=TARGET_SIZE):
    """
    Full pipeline: detect gray→ normalize→ CLAHE→ denoise→ ensure 3ch→ letterbox.
    """
    if img.ndim == 3 and img.shape[2] == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img.copy()

    norm = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')

    if clahe_cfg:
        norm = apply_clahe(norm, **clahe_cfg)

    if denoise_cfg:
        norm = denoise_image(norm, **denoise_cfg)

    bgr = cv2.cvtColor(norm, cv2.COLOR_GRAY2BGR)

    return letterbox_resize(bgr, new_shape=target_size)


In [26]:
# Batch Processing Function

def process_directory(input_dir, output_dir, exts=EXTS):
    """
    Walks through input_dir, preprocesses each image, and writes
    to output_dir mirroring folder structure.
    """
    total = 0
    for root, _, files in os.walk(input_dir):
        rel = os.path.relpath(root, input_dir)
        out_subdir = os.path.join(output_dir, rel)
        os.makedirs(out_subdir, exist_ok=True)

        for fname in files:
            if not any(fname.lower().endswith(ext) for ext in exts):
                continue
            src = os.path.join(root, fname)
            img = cv2.imread(src)
            if img is None:
                continue

            pre = preprocess_image(img)
            dst = os.path.join(out_subdir, fname)
            cv2.imwrite(dst, pre)
            total += 1

    return total


In [27]:
# Run Preprocessing

print(f"Preprocessing {INPUT_DIR} → {PREPROCESSED_OUTPUT_DIR} ...")
count = process_directory(INPUT_DIR, PREPROCESSED_OUTPUT_DIR)
print(f"Done! Processed {count} images.")


Preprocessing /Users/kereminci/Desktop/cms-team/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/train → /Users/kereminci/Desktop/cms-team/BYU_Locating_Bacterial_Flagellar_Motors_2025/data/preprocessed_data ...
Done! Processed 800 images.


Convert raw sample data to yolo format

[Object Detection dataset overview](https://docs.ultralytics.com/datasets/detect/)

In [28]:
# Creates dataset to fit yolo format

random.seed(42)

data_root    = PREPROCESSED_OUTPUT_DIR
labels_csv   = config.TRAIN_LABELS_PATH

# Output base paths (all under PROJECT_ROOT/data/yolo)
output_base         = os.path.join(config.PROJECT_ROOT, "data", "yolo")
output_images_train = os.path.join(output_base, "images", "train")
output_images_val   = os.path.join(output_base, "images", "val")
output_labels_train = os.path.join(output_base, "labels", "train")
output_labels_val   = os.path.join(output_base, "labels", "val")

for path in (output_images_train, output_images_val, output_labels_train, output_labels_val):
    os.makedirs(path, exist_ok=True)

labels_df = pd.read_csv(labels_csv)

positive_samples = {}
for _, row in labels_df.iterrows():
    if row["Motor axis 0"] == -1:  # -1 means no motor (negative sample)
        continue
    tomo_id    = row["tomo_id"]
    z          = int(row["Motor axis 0"])
    y          = row["Motor axis 1"]
    x          = row["Motor axis 2"]
    img_width  = row["Array shape (axis 2)"]
    img_height = row["Array shape (axis 1)"]
    img_name   = f"slice_{z:04d}.jpg"
    img_path   = os.path.join(data_root, tomo_id, img_name)
    positive_samples[img_path] = (x, y, img_width, img_height)

all_images = []
for root, _, files in os.walk(data_root):
    for fname in files:
        if fname.endswith(".jpg"):
            all_images.append(os.path.join(root, fname))

random.shuffle(all_images)
split_idx    = int(len(all_images) * 0.8)
train_images = all_images[:split_idx]
val_images   = all_images[split_idx:]

def process_images(image_list, img_out_dir, lbl_out_dir):
    for img_path in image_list:
        tomo_id = os.path.basename(os.path.dirname(img_path))
        z       = int(os.path.splitext(img_path)[0].split("_")[-1])
        base_fn = f"{tomo_id}_slice_{z:04d}"
        dest_img = os.path.join(img_out_dir,  f"{base_fn}.jpg")
        dest_lbl = os.path.join(lbl_out_dir, f"{base_fn}.txt")

        shutil.copyfile(img_path, dest_img)

        if img_path in positive_samples:
            x, y, w, h = positive_samples[img_path]
            bbox_w, bbox_h = 10, 10
            cx = x / w
            cy = y / h
            nw = bbox_w / w
            nh = bbox_h / h
            with open(dest_lbl, "w") as f:
                f.write(f"0 {cx} {cy} {nw} {nh}\n")
        else:
            open(dest_lbl, "w").close()

# Process both sets
process_images(train_images, output_images_train, output_labels_train)
process_images(val_images,   output_images_val,   output_labels_val)

print(f"Dataset created with {len(train_images)} training and {len(val_images)} validation images.")


Dataset created with 640 training and 160 validation images.


Data augmentation for training data using albumentations liblary.

In [29]:

# Data augmentation

# 1) Defines augmentation pipeline
transform = A.Compose([
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2,
                       rotate_limit=15, p=0.8),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2,
                               contrast_limit=0.2, p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
],

bbox_params=A.BboxParams(format='yolo',
                         label_fields=['class_labels'],
                         min_visibility=0.2)
)

def load_yolo_labels(label_path):
    bboxes, class_labels = [], []
    for line in open(label_path):
        cls, x, y, w, h = map(float, line.split())
        bboxes.append([x, y, w, h])
        class_labels.append(int(cls))
    return bboxes, class_labels

# Paths for augmented data output and yolo format input
data_root    = config.YOLO_DATA_DIR
output_root = config.AUGMENTED_YOLO_DATA

IMAGES_DIR = os.path.join(data_root, "images", "train")
LABELS_DIR = os.path.join(data_root, "labels", "train")

os.path.join(output_root, "images", "train")
os.path.join(output_root, "labels", "train")

OUT_IMAGES = os.path.join(output_root, "images", "train")
OUT_LABELS = os.path.join(output_root, "labels", "train")

os.makedirs(OUT_IMAGES, exist_ok=True)
os.makedirs(OUT_LABELS, exist_ok=True)

for img_name in tqdm(os.listdir(IMAGES_DIR)):
    img_path = os.path.join(IMAGES_DIR, img_name)
    label_path = os.path.join(LABELS_DIR, img_name.replace('.jpg','.txt'))

    img = cv2.imread(img_path)
    bboxes, class_labels = load_yolo_labels(label_path)

    # apply N random augmentations per image
    for i in range(3):  # generates 3 aug versions
        transformed = transform(image=img, bboxes=bboxes,
                                class_labels=class_labels)
        aug_img  = transformed['image']
        aug_bboxes = transformed['bboxes']
        aug_labels = transformed['class_labels']

        out_img = os.path.join(OUT_IMAGES, f"{os.path.splitext(img_name)[0]}_aug{i}.jpg")
        cv2.imwrite(out_img, aug_img)

        out_lbl = os.path.join(OUT_LABELS, f"{os.path.splitext(img_name)[0]}_aug{i}.txt")
        with open(out_lbl, 'w') as f:
            for cls, (x,y,w,h) in zip(aug_labels, aug_bboxes):
                f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")


/Users/kereminci/Desktop/cms-team/BYU_Locating_Bacterial_Flagellar_Motors_2025/venv/lib/python3.13/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/var/folders/3c/s9rrk0m14p5b80qglgftm0900000gn/T/ipykernel_87471/2678116614.py:7: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
/var/folders/3c/s9rrk0m14p5b80qglgftm0900000gn/T/ipykernel_87471/2678116614.py:12: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
100%|██████████| 640/640 [00:55<00:00, 11.55it/s]
